In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("STEP 1: Input parameters")
print("=" * 80)
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'")
print(f"  Source table      : adb_caresync.bronze.{table_name}")
print(f"  Target table      : adb_caresync.silver.{table_name}")

In [0]:
source_table = f"adb_caresync.bronze.{table_name}"
filter_condition = f"landing_TimeStamp = '{landing_timestamp}'"

print("=" * 80)
print("STEP 2: Reading data from Bronze")
print("=" * 80)
print("Equivalent query:")
print(f"  SELECT * FROM {source_table}\n  WHERE {filter_condition}")

bronze_df = spark.read.table(source_table).filter(filter_condition)

bronze_count = bronze_df.count()
print(f"\nRecords read from Bronze: {bronze_count}")
print(f"Columns ({len(bronze_df.columns)}): {bronze_df.columns}")
if bronze_count == 0:
    print("WARNING: No records found for this landing_timestamp. Check the parameter value.")

In [0]:
from pyspark.sql import functions as F

print("=" * 80)
print("STEP 3: Adding audit columns")
print("=" * 80)
print("  insert_timestamp = current_timestamp()")
print("  update_timestamp = current_timestamp()")

updated_df = bronze_df.withColumn("insert_timestamp",F.current_timestamp()).withColumn("update_timestamp", F.current_timestamp())

print(f"\nColumns after transformation ({len(updated_df.columns)}): {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresync.silver.{table_name}"
load_type_upper = load_type.strip().upper()
cols_sql = ", ".join([f"`{c}`" for c in updated_df.columns])

print("=" * 80)
print(f"STEP 4: Writing to Silver | load_type = {load_type_upper}")
print("=" * 80)
print(f"Target table: {target_table}")
target_exists = spark.catalog.tableExists(target_table)
print(f"Target table exists: {target_exists}")
if target_exists:
    print(f"Target record count BEFORE load: {spark.table(target_table).count()}")

if load_type_upper == "FULL":
    # Full load: overwrite the target table
    print("\nFULL load -> target table will be overwritten.")
    print("Equivalent query:")
    print(f"  CREATE OR REPLACE TABLE {target_table} AS\n"
          f"  SELECT {cols_sql}\n"
          f"  FROM <bronze_data_with_audit_columns>")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print("FULL load completed.")

elif load_type_upper == "APPEND":
    # Append: add incoming records to the target table (creates it if missing)
    print("\nAPPEND load -> records will be appended to the target table.")
    print("Equivalent query:")
    print(f"  INSERT INTO {target_table} ({cols_sql})\n"
          f"  SELECT {cols_sql}\n"
          f"  FROM <bronze_data_with_audit_columns>")
    updated_df.write.mode("append").saveAsTable(target_table)
    print("APPEND load completed.")

elif load_type_upper == "MERGE":
    # Merge: update existing records and insert new ones based on merge keys
    print("\nMERGE load -> matching records will be updated, new records inserted.")
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"Merge keys: {keys}")
    if not keys:
        raise ValueError(
            "merge_keys must be provided (comma-separated) when load_type is MERGE"
        )
    missing = [k for k in keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source data: {missing}")

    # Avoid multiple source rows matching the same target row
    source_df = updated_df.dropDuplicates(keys)
    incoming_count = updated_df.count()
    source_count = source_df.count()
    print(f"Incoming records: {incoming_count} | after de-duplication on merge keys: "
          f"{source_count} (duplicates removed: {incoming_count - source_count})")

    if not target_exists:
        # First run: create the target table
        print("\nTarget table does not exist -> creating it from source data (initial load).")
        print("Equivalent query:")
        print(f"  CREATE TABLE {target_table} AS\n"
              f"  SELECT {cols_sql}\n"
              f"  FROM <bronze_data_with_audit_columns>")
        source_df.write.mode("overwrite").saveAsTable(target_table)
        print("Initial load completed.")
    else:
        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])
        # On update, keep the original insert_timestamp
        update_set = {
            f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"
        }
        insert_values = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        print(f"\nMerge condition: {merge_condition}")
        print("Columns updated on match (insert_timestamp preserved): "
              f"{[c for c in source_df.columns if c != 'insert_timestamp']}")

        update_sql = ",\n      ".join([f"t.{k} = {v}" for k, v in update_set.items()])
        insert_cols_sql = ", ".join(insert_values.keys())
        insert_vals_sql = ", ".join(insert_values.values())
        print("\nEquivalent MERGE query:")
        print(f"  MERGE INTO {target_table} AS t\n"
              f"  USING <bronze_data_with_audit_columns> AS s\n"
              f"  ON {merge_condition}\n"
              f"  WHEN MATCHED THEN UPDATE SET\n      {update_sql}\n"
              f"  WHEN NOT MATCHED THEN INSERT ({insert_cols_sql})\n"
              f"  VALUES ({insert_vals_sql})")

        print("\nExecuting MERGE...")
        (
            DeltaTable.forName(spark, target_table)
            .alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_values)
            .execute()
        )

        # Show merge metrics from the latest Delta history entry
        metrics = (
            DeltaTable.forName(spark, target_table)
            .history(1)
            .select("operationMetrics")
            .collect()[0][0]
        ) or {}
        print("MERGE completed. Metrics:")
        print(f"  Source rows    : {metrics.get('numSourceRows')}")
        print(f"  Rows updated   : {metrics.get('numTargetRowsUpdated')}")
        print(f"  Rows inserted  : {metrics.get('numTargetRowsInserted')}")
        print(f"  Rows unchanged : {metrics.get('numTargetRowsCopied')}")

else:
    print(f"ERROR: Unsupported load_type '{load_type}'")
    raise ValueError(
        f"Unsupported load_type '{load_type}'. Expected FULL, APPEND or MERGE."
    )

print(f"\nTarget record count AFTER load: {spark.table(target_table).count()}")

In [0]:
%skip
%sql
SELECT * FROM adb_caresync.silver.hospitals


In [0]:
record_count = updated_df.count()
print("=" * 80)
print(f"STEP 5: Finished. Records processed: {record_count}. Exiting notebook.")
print("=" * 80)
dbutils.notebook.exit(str(record_count))